# Norway vs. US Labor Market — Exploratory Analysis

This notebook walks through the data before building the dashboard.
The goal is to understand what we're working with, spot any data quality issues,
and identify the key findings worth highlighting.

**Before running:** make sure `data/processed/` has the CSV files.
If not, run `python generate_sample_data.py` from the project root.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os

PROCESSED_DIR = "../data/processed"

# Load the three clean datasets
unemployment = pd.read_csv(f"{PROCESSED_DIR}/unemployment_clean.csv")
wages        = pd.read_csv(f"{PROCESSED_DIR}/wages_clean.csv")
employment   = pd.read_csv(f"{PROCESSED_DIR}/employment_clean.csv")

print("unemployment:", unemployment.shape)
print("wages:",        wages.shape)
print("employment:",   employment.shape)

unemployment: (30, 3)
wages: (60, 7)
employment: (90, 6)


## 1. Data Overview

Check shapes, column names, data types, and missing values before doing any analysis.

In [2]:
print("=== Unemployment ===")
print(unemployment.dtypes)
print("\nMissing values:")
print(unemployment.isnull().sum())
print("\nYear range:", unemployment['year'].min(), "to", unemployment['year'].max())
unemployment.head(8)

=== Unemployment ===
year                   int64
unemployment_rate    float64
country               object
dtype: object

Missing values:
year                 0
unemployment_rate    0
country              0
dtype: int64

Year range: 2010 to 2024


,year,unemployment_rate,country
0,2010,3.8,Norway
1,2011,3.4,Norway
2,2012,3.3,Norway
3,2013,3.8,Norway
4,2014,3.6,Norway
5,2015,4.5,Norway
6,2016,4.7,Norway
7,2017,4.2,Norway


In [3]:
print("=== Wages ===")
print(wages.dtypes)
print("\nIndustries covered:", wages['industry'].unique())
print("\nMissing wage values:", wages['wage_annual_usd_ppp'].isnull().sum())
wages[wages['industry'] == 'Technology'].describe()

=== Wages ===
country                 object
year                     int64
industry_code           object
industry                object
wage_local             float64
wage_local_currency     object
wage_annual_usd_ppp    float64
dtype: object

Industries covered: ['Technology' 'Professional Services' 'Total']

Missing wage values: 0


,year,wage_local,wage_annual_usd_ppp
count,30.000000,30.000000,30.000000
mean,2017.000000,331761.418111,72016.866667
std,4.394354,343926.764649,14063.326079
min,2010.000000,30.560000,53383.000000
25%,2013.250000,38.723542,62615.000000
50%,2017.000000,259733.540833,68305.000000
75%,2020.750000,650700.000000,78819.750000
max,2024.000000,847065.000000,104170.000000


In [4]:
print("=== Employment ===")
print(employment['industry'].value_counts())
employment[employment['industry'] == 'Technology']

=== Employment ===
industry
Technology               30
Finance                  15
Manufacturing            15
Professional Services    15
Total                    15
Name: count, dtype: int64


,country,year,industry_code,industry,employment_count,employment_pct
1,Norway,2010,Information and communcation,Technology,8.580000e+04,3.36
6,Norway,2011,Information and communcation,Technology,8.700000e+04,3.36
11,Norway,2012,Information and communcation,Technology,8.860000e+04,3.35
16,Norway,2013,Information and communcation,Technology,8.960000e+04,3.35
21,Norway,2014,Information and communcation,Technology,9.180000e+04,3.40
26,Norway,2015,Information and communcation,Technology,9.310000e+04,3.43
31,Norway,2016,Information and communcation,Technology,9.260000e+04,3.40
36,Norway,2017,Information and communcation,Technology,9.380000e+04,3.41
41,Norway,2018,Information and communcation,Technology,9.710000e+04,3.47
46,Norway,2019,Information and communcation,Technology,1.020000e+05,3.59


## 2. Key Finding 1: Unemployment Resilience

Norway's unemployment rate stayed much lower than the US throughout 2010-2024,
including during the COVID-19 shock. 

Hypothesis: Norway's active labor market policies (keeping people in employment
through subsidies) and generous unemployment benefits cushioned the shock
compared to the US layoff-heavy model.

In [5]:
fig = px.line(
    unemployment,
    x='year', y='unemployment_rate', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Annual Unemployment Rate: Norway vs. United States (2010-2024)',
    labels={'unemployment_rate': 'Unemployment Rate (%)', 'year': 'Year'}
)
fig.add_vrect(x0=2020, x1=2021, fillcolor='rgba(255,200,0,0.15)',
              line_width=0, annotation_text='COVID-19')
fig.update_layout(yaxis_ticksuffix='%', hovermode='x unified')
fig.show()

# Quantify the COVID gap
covid_year = unemployment[unemployment['year'] == 2020].set_index('country')['unemployment_rate']
print(f"\n2020 unemployment rates:")
print(covid_year)
print(f"US-Norway gap in 2020: {covid_year['United States'] - covid_year['Norway']:.1f} percentage points")


2020 unemployment rates:
country
Norway           4.6
United States    8.1
Name: unemployment_rate, dtype: float64
US-Norway gap in 2020: 3.5 percentage points


## 3. Key Finding 2: Tech Wages (PPP-Adjusted)

Are US tech workers paid more or less than Norwegian tech workers once we account
for cost-of-living differences?

We use OECD PPP conversion factors to normalize wages to internationally
comparable 'international dollars.'

In [6]:
tech_wages = wages[wages['industry'] == 'Technology'].copy()

fig = px.line(
    tech_wages,
    x='year', y='wage_annual_usd_ppp', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Sector Annual Wages, PPP-Adjusted USD (2010-2024)',
    labels={'wage_annual_usd_ppp': 'Annual Wage (PPP USD)', 'year': 'Year'}
)
fig.update_layout(yaxis_tickprefix='$', yaxis_tickformat=',')
fig.show()

# 2023 comparison
wages_2023 = tech_wages[tech_wages['year'] == 2023].set_index('country')['wage_annual_usd_ppp']
print("\n2023 tech wages (PPP-adjusted annual USD):")
print(wages_2023)
if 'United States' in wages_2023.index and 'Norway' in wages_2023.index:
    diff_pct = (wages_2023['United States'] / wages_2023['Norway'] - 1) * 100
    print(f"\nUS tech wages are {diff_pct:.1f}% {'higher' if diff_pct > 0 else 'lower'} than Norway (PPP-adjusted)")


2023 tech wages (PPP-adjusted annual USD):
country
Norway            71658.0
United States    100658.0
Name: wage_annual_usd_ppp, dtype: float64

US tech wages are 40.5% higher than Norway (PPP-adjusted)


## 4. Key Finding 3: The Tech Wage Premium

How much more do tech workers earn compared to the national average in each country?
A higher premium suggests tech is a more 'elite' sector in that economy.

In [7]:
tech_w  = wages[wages['industry'] == 'Technology'][['country','year','wage_annual_usd_ppp']].rename(columns={'wage_annual_usd_ppp': 'tech_wage'})
total_w = wages[wages['industry'] == 'Total'][['country','year','wage_annual_usd_ppp']].rename(columns={'wage_annual_usd_ppp': 'avg_wage'})

merged = tech_w.merge(total_w, on=['country','year'])
merged['premium_ratio'] = (merged['tech_wage'] / merged['avg_wage']).round(2)

fig = px.line(
    merged,
    x='year', y='premium_ratio', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Wage Premium (Tech Wage / National Average Wage)',
    labels={'premium_ratio': 'Premium Ratio', 'year': 'Year'}
)
fig.add_hline(y=1.0, line_dash='dash', line_color='gray', annotation_text='National average')
fig.show()

print("\n2023 tech wage premium:")
print(merged[merged['year']==2023][['country','tech_wage','avg_wage','premium_ratio']].to_string(index=False))


2023 tech wage premium:
country  tech_wage  avg_wage  premium_ratio
 Norway    71658.0   49976.0           1.43


## 5. Tech Employment Share

What percentage of each country's workforce works in tech?

In [8]:
tech_emp = employment[employment['industry'] == 'Technology'].copy()

fig = px.line(
    tech_emp,
    x='year', y='employment_pct', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Tech Sector as % of Total Workforce (2010-2024)',
    labels={'employment_pct': '% of Total Workforce', 'year': 'Year'}
)
fig.update_layout(yaxis_ticksuffix='%')
fig.show()

# Latest year
latest = tech_emp['year'].max()
print(f"\n{latest} tech employment share:")
print(tech_emp[tech_emp['year']==latest][['country','employment_pct','employment_count']].to_string(index=False))


2024 tech employment share:
      country  employment_pct  employment_count
       Norway            3.97      1.189000e+05
United States            1.86      2.922333e+06


## 6. Limitations

Every analysis has limitations. Documenting them shows analytical maturity.

1. **Industry classification mismatch:** Norway's NACE J (Information & Communication)
   includes publishing and broadcasting, while the US NAICS 51 (Information) has a
   slightly different scope. Comparisons are indicative, not exact.

2. **PPP assumptions:** We use OECD PPP factors for GDP. Tech workers' actual
   purchasing power might differ from economy-wide averages (they may live in
   more expensive cities). PPP is an approximation.

3. **Wage data coverage:** SSB wages cover full-time employees in registered
   enterprises. BLS wages cover production and nonsupervisory workers in some
   series. Gig workers and self-employed are not fully captured in either.

4. **Time granularity:** We aggregate monthly BLS data to annual averages.
   This smooths out seasonal variation but also hides the peak of COVID
   unemployment (April 2020: 14.7% in the US) in the annual average.

5. **Data availability:** Some SSB tables only go back to 2011 or 2015 for
   certain industry breakdowns. Where data is unavailable, we document it
   rather than interpolate.

In [ ]:
forecast_years = np.array([2025, 2026, 2027, 2028])
fig_fc = go.Figure()

for country, color in [('Norway', '#003087'), ('United States', '#B22234')]:
    df = wages[(wages['country'] == country) & (wages['industry'] == 'Technology')].sort_values('year')
    x, y = df['year'].values, df['wage_annual_usd_ppp'].values

    m, b = np.polyfit(x, y, 1)
    residuals = y - (m * x + b)
    se = np.std(residuals)

    all_x   = np.concatenate([x, forecast_years])
    fitted  = m * all_x + b
    upper   = fitted + 1.96 * se
    lower   = fitted - 1.96 * se

    # Historical line
    fig_fc.add_trace(go.Scatter(
        x=x, y=y, mode='lines+markers', name=country,
        line=dict(color=color, width=2.5), marker=dict(size=7)
    ))
    # Forecast line (dashed)
    fig_fc.add_trace(go.Scatter(
        x=np.concatenate([[x[-1]], forecast_years]),
        y=np.concatenate([[y[-1]], m * forecast_years + b]),
        mode='lines', name=f'{country} forecast',
        line=dict(color=color, dash='dash', width=2), showlegend=True
    ))
    # Confidence band (forecast years only)
    fc_fitted = m * forecast_years + b
    fc_upper  = fc_fitted + 1.96 * se
    fc_lower  = fc_fitted - 1.96 * se
    fig_fc.add_trace(go.Scatter(
        x=np.concatenate([forecast_years, forecast_years[::-1]]),
        y=np.concatenate([fc_upper, fc_lower[::-1]]),
        fill='toself', fillcolor=color.replace('#', 'rgba(') + ',0.12)' if False else f'rgba({int(color[1:3],16)},{int(color[3:5],16)},{int(color[5:7],16)},0.12)',
        line=dict(width=0), showlegend=False, name=f'{country} 95% band'
    ))

    print(f"{country}: slope = ${m:,.0f}/yr | 2028 forecast = ${m*2028+b:,.0f} (±${1.96*se:,.0f})")

fig_fc.add_vline(x=2024, line_dash='dot', line_color='gray',
                 annotation_text='forecast start', annotation_position='top right')
fig_fc.update_layout(
    title='Tech Wage Forecast to 2028 — Linear Trend with 95% Prediction Band',
    yaxis_tickprefix='$', yaxis_tickformat=',',
    xaxis_title='Year', yaxis_title='Annual Wage (PPP-Adjusted USD)',
    template='plotly_white', hovermode='x unified', height=500
)
fig_fc.show()

## 9. Simple Wage Forecast (2025–2028)

The data shows a clear linear trend in tech wages for both countries.
A simple question: if current growth rates continue, where does the gap end up?

This is a linear extrapolation — not a sophisticated model — but it shows
the direction and quantifies the trajectory. I'm using:
- `numpy.polyfit` to fit a linear trend to 2010–2024 data
- Residual standard deviation to build a 95% prediction band
- The band widens further out (more uncertainty the further we forecast)

**Limitations of this approach:**
- Assumes the linear trend continues (no structural breaks, no tech recession)
- A 4-year forecast horizon is about the limit for a simple linear model
- The PPP factors themselves might shift, which isn't captured here

In [ ]:
# Compute the DiD estimate
pre  = unemployment[unemployment['year'].between(2015, 2019)].groupby('country')['unemployment_rate'].mean()
post = unemployment[unemployment['year'].between(2020, 2022)].groupby('country')['unemployment_rate'].mean()

norway_delta = post['Norway'] - pre['Norway']
us_delta     = post['United States'] - pre['United States']
did          = norway_delta - us_delta

print("Pre-COVID average unemployment (2015–2019):")
print(pre.round(2))
print("\nCOVID period average (2020–2022):")
print(post.round(2))
print(f"\nNorway Δ: {norway_delta:+.2f} pp")
print(f"US Δ:     {us_delta:+.2f} pp")
print(f"\nDiD estimate: {did:+.2f} pp")
print(
    f"\nInterpretation: Norway's institutions absorbed an estimated {abs(did):.2f} pp "
    f"of unemployment shock relative to the US."
)

In [ ]:
# Check parallel trends assumption first — plot 2015-2019 indexed to 100 in 2015
pre_trend = unemployment[unemployment['year'].between(2015, 2019)].copy()
base_2015 = pre_trend[pre_trend['year'] == 2015].set_index('country')['unemployment_rate']
pre_trend['indexed'] = pre_trend.apply(
    lambda r: round(r['unemployment_rate'] / base_2015[r['country']] * 100, 1), axis=1
)

fig = px.line(
    pre_trend, x='year', y='indexed', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    markers=True,
    title='Parallel Trends Check: Pre-COVID Unemployment (2015 = 100)',
    labels={'indexed': 'Unemployment Index (2015 = 100)', 'year': 'Year'}
)
fig.add_hline(y=100, line_dash='dash', line_color='gray')
fig.show()

# If lines move together -> assumption holds
# Norway went from 100 to ~82 (declining), US went from 100 to ~70 (also declining)
# Similar downward direction -> parallel trends is plausible

## 8. Difference-in-Differences Setup

The correlation analysis showed we can't just look at associations — we need
a causal framework. COVID-19 creates a natural experiment:

- **Treatment group:** Norway — strong social safety net, wage subsidy schemes,
  active labor market policies
- **Control group:** United States — weaker safety net, primarily layoff-based
  adjustment
- **Shock:** COVID-19 hit both countries at the same time, same year (2020)

**DiD logic:**
If both countries would have followed similar unemployment paths without COVID
(parallel trends assumption), then the *difference in how much each country's
unemployment changed* estimates the causal effect of labor market institutions.

`DiD = (Norway_post - Norway_pre) - (US_post - US_pre)`

**What's actually happening:**

Looking at the table, the US tech employment share was *declining* from 2.09% (2010)
down to around 1.86% (2024) — not growing. Meanwhile unemployment was also declining
over the same period (from 9.6% to ~4%).

So both variables trended downward together over 2010–2024, creating a spurious
positive correlation. High-tech-share years (2010–2012) happened to coincide with
high unemployment (post-financial crisis). Low-tech-share years (2019–2024) coincided
with low unemployment (recovery and expansion).

This is a classic **time series spurious correlation** — two variables that share
a common downward trend will appear positively correlated even if they have no
real relationship with each other.

For Norway, the tech share was *growing* (3.36% → 3.97%) while unemployment
stayed relatively flat and low, so there's no strong common trend and the
correlation is near zero.

**Takeaway:** The correlation result is not meaningful on its own — it's picking
up the post-2008 recovery trend, not a genuine relationship between tech employment
and unemployment. This is why causality requires DiD or regression, not just correlation.

In [ ]:
# Look at what the US tech employment share actually did over time
us_data = corr_df[corr_df['country'] == 'United States'].sort_values('year')
print("US tech employment % by year:")
print(us_data[['year','employment_pct','unemployment_rate']].to_string(index=False))

In [ ]:
from scipy import stats

tech_emp = employment[employment['industry'] == 'Technology'][['country','year','employment_pct']]
corr_df  = tech_emp.merge(unemployment, on=['country','year'])

for country in ['Norway', 'United States']:
    df = corr_df[corr_df['country'] == country]
    r, p = stats.pearsonr(df['employment_pct'], df['unemployment_rate'])
    print(f"{country}: r = {r:.2f}, p = {p:.3f}")

# Plot it — does anything look off?
fig = px.scatter(
    corr_df, x='employment_pct', y='unemployment_rate', color='country',
    color_discrete_map={'Norway': '#003087', 'United States': '#B22234'},
    hover_data=['year'],
    title='Tech Employment Share vs. Unemployment Rate (each point = 1 year)',
    labels={'employment_pct': 'Tech Employment (% of workforce)',
            'unemployment_rate': 'Unemployment Rate (%)'}
)
fig.update_traces(marker=dict(size=9))
fig.show()

## 7. Something That Surprised Me — The US Correlation

Before building the DiD, I ran a quick correlation between tech employment share
and unemployment rate for each country. My hypothesis was: more tech workers =
lower unemployment (tech creates jobs, healthy economy, etc.).

Norway matched the hypothesis — but the US didn't. The US showed a *positive*
correlation (r ≈ 0.66), meaning higher tech employment share years coincided
with *higher* unemployment. That seemed wrong at first.

Let me dig into why.